In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs_Lam0LamCUpdated.parquet")

print(list(field for field in data.fields if 'LambdaC' in field))

mothIdx = data['mothIdx']
mcLund = data['mcLund']
pridx = data['pMCIdx']

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array([int(s, 2) for s in binrep], dtype=np.int64)
    #tempbits = np.array([list(s) for s in binrep]).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):
    bit_to_look_for = pid_map_object.selectors.index(selector)
    place = 1 << bit_to_look_for          # 2**bit_to_look_for, was 10**bit_to_look_for
    mask = (bits // place) % 2            # was % 10
    mask_bool = ak.values_astype(mask, bool)
    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    Ld1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    Ld2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]

    LCd1idx = ak_arr.LambdaCd1Idx[ak_arr.Bd1Idx]
    LCd2idx = ak_arr.LambdaCd2Idx[ak_arr.Bd1Idx]
    LCd3idx = ak_arr.LambdaCd3Idx[ak_arr.Bd1Idx]
    LCd4idx = ak_arr.LambdaCd4Idx[ak_arr.Bd1Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    trkidx_kaon = ak_arr['KTrkIdx']
    trk_selector_map_kaon = ak_arr['KSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['Ld1idx'] = Ld1idx
    indices_and_maps['Ld2idx'] = Ld2idx
    indices_and_maps['LCd1idx'] = LCd1idx
    indices_and_maps['LCd2idx'] = LCd2idx
    indices_and_maps['LCd3idx'] = LCd3idx
    indices_and_maps['LCd4idx'] = LCd4idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    indices_and_maps['trkidx_kaon'] = trkidx_kaon
    indices_and_maps['trk_selector_map_kaon'] = trk_selector_map_kaon

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lamp_selector,\
              lampi_selector, \
              lamCp_selector,\
              lamCpi_selector,\
              lamCk_selector,\
             verbosity=0):
                #change Bpselector
    Ld1idx = indices_and_maps['Ld1idx']
    Ld2idx = indices_and_maps['Ld2idx']
    LCd1idx = indices_and_maps['LCd1idx']
    LCd2idx = indices_and_maps['LCd2idx']
    LCd3idx = indices_and_maps['LCd3idx']
    LCd4idx = indices_and_maps['LCd4idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']

    trkidx_kaon = indices_and_maps['trkidx_kaon']
    trk_selector_map_kaon = indices_and_maps['trk_selector_map_kaon']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)

    kabits = calculate_bits_for_PID_selector(trkidx_kaon, trk_selector_map_kaon, verbose=verbosity)
    
    mask_bool_protonL = mask_PID_selection(pbits[Ld1idx], lamp_selector, pps)
        
    mask_bool_pionL = mask_PID_selection(pibits[Ld2idx], lampi_selector, pips)
    

    lcList = []
    for i in range(len(LCd1idx)):
        #print("I",i)
        for val in LCd1idx[i]:
            #print("VAL", val)
            if len(pridx[i])>val:
                if abs(mcLund[i][mothIdx[i][pridx[i][val]]]) == 4122:
                    lcList += [val]
                    break
            else:
                lcList+=[0]
                break

        if len(lcList)==i:
            lcList += [0]

    lc2ListPi = []
    lc2ListKa = []
    for i in range(len(LCd2idx)):
        #print("I",i)
        for val in LCd2idx[i]:
            #print("VAL", val)
            if len(pridx[i])>val:
                if abs(mcLund[i][mothIdx[i][pridx[i][val]]]) == 4122:
                    if abs(mcLund[i][pridx[i][val]]) == 321:
                        lc2ListKa += [val]
                    elif abs(mcLund[i][pridx[i][val]]) == 211: 
                        lc2ListPi += [val]
                    if len(lc2ListKa)==i+1 and len(lc2ListPi)==i+1:
                        break
            else:
                lc2ListKa+=[0]
                lc2ListPi+=[0]
                break

        if len(lc2ListKa)==i:
            lc2ListKa+=[0]
        if len(lc2ListPi)==i:
            lc2ListPi+=[0]

    #print(lcList)




    #LCd1idx = list(val[0] for val in LCd1idx)
    #LCd2idx = list(val[0] for val in LCd2idx)

    mask_bool_protonLC = mask_PID_selection(pbits[lcList], lamCp_selector, pps)

    mask_bool_kaonLC = mask_PID_selection(kabits[lc2ListKa], lamCk_selector, kaps)
        
    mask_bool_pionLC2 = mask_PID_selection(pibits[lc2ListPi], lamCpi_selector, pips)

    #works
    mask_bool_pionLC3 = mask_PID_selection(pibits[LCd3idx], lamCpi_selector, pips)

    #works
    mask_bool_pionLC4 = mask_PID_selection(pibits[LCd4idx], lamCpi_selector, pips)


    return mask_bool_protonL, mask_bool_pionL, mask_bool_protonLC, mask_bool_kaonLC, mask_bool_pionLC2, mask_bool_pionLC3, mask_bool_pionLC4


In [ ]:
lamCMass = data['LambdaC_unc_Mass']
lam0Mass = data['Lambda0_unc_Mass']
mcLund = data['mcLund']
mothIdx = data['mothIdx']
bMC = data['BMCIdx']
lam0FS = data['Lambda0postFitFlightSignificance']
ksFS = data['K_SpostFitFlightSignificance']
lamCnDau = data['LambdaC_unc_nDaus']
pmaskPick = 'SuperLooseKMProtonSelection'
pimaskPick = 'SuperLooseKMPionMicroSelection'
kamaskPick =  'SuperLooseKMKaonMicroSelection'

lamCMassLooseMask = (lamCMass>2.265912)&(lamCMass<2.299912)
lam0MassLooseMask = (lam0Mass>1.112683)&(lam0Mass<1.118683)
lam0FSLooseMask = (lam0FS>5)
ksFSLooseMask = (ksFS>5)



tmBInd=[]
for i in range(len(data)):
    mcLundi = mcLund[i]
    for val in bMC[i]:
        if abs(mcLundi[val])==521:
            tmBInd += [i]
            continue

print(len(tmBInd), len(data))


In [ ]:
dd = {}
d1L = []
d1Lund = data['LambdaCd1Lund']
d2Lund = data['LambdaCd2Lund']
d3Lund = data['LambdaCd3Lund']
d4Lund = data['LambdaCd4Lund']
oC = tC = thC = fC = 0
for i in tmBInd:
    nDi = lamCnDau[i]
    if len(nDi)>1:
        nDi=nDi[0]

    d1L += list(d1Lund[i])


    if nDi==2:
        if d3Lund[i][0] == 0:
            dd[i]=2
            tC+=1
    elif nDi==3:
        if d3Lund[i][0] == 0:
            dd[i]=2
            tC+=1
        elif abs(d2Lund[i][0]) == 321:
            dd[i]=1
            oC+=1
    else:
        if len(lam0FSLooseMask[i])>1 and abs(d2Lund[i][0])==211:
            dd[i]=4
            fC+=1
        else:
            dd[i]=3
            thC+=1

print("DM1, tmB events:", oC)
print("DM2, tmB events:", tC)
print("DM3, tmB events:", thC)
print("DM4, tmB events:", fC)

In [ ]:
dm1lamCcount = dm2lamCcount = dm3lamCcount = dm4lamCcount = 0
for i in tmBInd:
    if dd[i] == 1:
        dm1lamCcount+=len(lamCMassLooseMask[i])
    elif dd[i] == 2:
        dm2lamCcount+=len(lamCMassLooseMask[i])
    elif dd[i] == 3:
        dm3lamCcount+=len(lamCMassLooseMask[i])
    elif dd[i] == 4:
        dm4lamCcount+=len(lamCMassLooseMask[i])

print("dm1, tmB, lamC cand:", dm1lamCcount)
print("dm2, tmB, lamC cand:", dm2lamCcount)
print("dm3, tmB, lamC cand:", dm3lamCcount)
print("dm4, tmB, lamC cand:", dm4lamCcount)

In [ ]:
lam0MassWinDprecut = {1: 0, 2: 0, 3: 0, 4: 0}
for i in tmBInd:
    lam0MassWinDprecut[dd[i]]+=len(lam0MassLooseMask[i])

print("dm1, tmB, lam0 cand:", lam0MassWinDprecut[1])
print("dm2, tmB, lam0 cand:", lam0MassWinDprecut[2])
print("dm3, tmB, lam0 cand:", lam0MassWinDprecut[3])
print("dm4, tmB, lam0 cand:", lam0MassWinDprecut[4])

In [ ]:
lamCMassCut = lamCMass[lamCMassLooseMask]

lamCMassWinD = {1: 0, 2: 0, 3: 0, 4: 0}
for i in tmBInd:
    lamCMassWinD[dd[i]]+=len(lamCMassCut[i])

print("dm1, tmB, lamC cand:", lamCMassWinD[1])
print("dm2, tmB, lamC cand:", lamCMassWinD[2])
print("dm3, tmB, lamC cand:", lamCMassWinD[3])
print("dm4, tmB, lamC cand:", lamCMassWinD[4])


In [ ]:
lam0MassCut = lam0Mass[lam0MassLooseMask]
lam0MassWinD = {1: 0, 2: 0, 3: 0, 4: 0}
for i in tmBInd:
    lam0MassWinD[dd[i]]+=len(lam0MassCut[i])

print("dm1, tmB, lam0 cand:", lam0MassWinD[1])
print("dm2, tmB, lam0 cand:", lam0MassWinD[2])
print("dm3, tmB, lam0 cand:", lam0MassWinD[3])
print("dm4, tmB, lam0 cand:", lam0MassWinD[4])

In [ ]:
lam0comb = lam0FSLooseMask&lam0MassLooseMask
#lam0MassFSCut = lam0Mass[lam0comb]
lam0MassWinDcomb = {1: 0, 2: 0, 3: 0, 4: 0}
for i in tmBInd:
    lam0MassWinDcomb[dd[i]]+=ak.sum(lam0comb[i])

print("dm1, tmB, lam0 cand:", lam0MassWinDcomb[1])
print("dm2, tmB, lam0 cand:", lam0MassWinDcomb[2])
print("dm3, tmB, lam0 cand:", lam0MassWinDcomb[3])
print("dm4, tmB, lam0 cand:", lam0MassWinDcomb[4])

In [ ]:
#ksFSCut = ksFS[ksFSLooseMask]
d2Idx = data['LambdaCd2Idx']

mask_list = [[False] * len(sub) for sub in ak.to_list(lamCMassLooseMask)]

for i in tmBInd:
    if dd[i]==2 or dd[i]==3:
        j = d2Idx[i][0]
        mask_list[i][j] = ksFSLooseMask[i][j]&lamCMassLooseMask[i][j]
    else:
        mask_list[i] = lamCMassLooseMask[i]

ksFScutMassMask = ak.Array(mask_list)
lamCCandksFSMassCutD={1:0,2:0,3:0,4:0}
for i in tmBInd:
    lamCCandksFSMassCutD[dd[i]] += ak.sum(ksFScutMassMask[i])

print("dm1, tmB, lamC cand:", lamCCandksFSMassCutD[1])
print("dm2, tmB, lamC cand:", lamCCandksFSMassCutD[2])
print("dm3, tmB, lamC cand:", lamCCandksFSMassCutD[3])
print("dm4, tmB, lamC cand:", lamCCandksFSMassCutD[4])



In [ ]:
def buildPidMask(pM, piM, kM):
    pbits = calculate_bits_for_PID_selector(data.pTrkIdx,data.pSelectorsMap)
    pibits = calculate_bits_for_PID_selector(data.piTrkIdx,data.piSelectorsMap)
    kabits = calculate_bits_for_PID_selector(data.KTrkIdx,data.KSelectorsMap)


    pps = myPIDselector.PIDselector("p")
    pips = myPIDselector.PIDselector("pi")
    kaps = myPIDselector.PIDselector("K")

    indexFailList = []

    pmask = mask_PID_selection(pbits,pM,pps)
    pimask = mask_PID_selection(pibits,piM,pips)
    kamask = mask_PID_selection(kabits,kM, kaps)

    pidMask = {}
    for i in tmBInd: 
        l1Idx = data.Bd1Idx[i,0]
        l2Idx = data.Bd2Idx[i,0]



        if pmask[i][data.Lambda0d1Idx[i][l2Idx]] == False:
            pidMask[i] = False
            continue

        if pimask[i][data.Lambda0d2Idx[i][l2Idx]] == False:
            pidMask[i] = False
            continue


        if abs(dd[i])!=4:
            if len(data.pTrkIdx[i])>data.LambdaCd1Idx[i][l1Idx]:
                if pmask[i][data.LambdaCd1Idx[i][l1Idx]] == False:
                    pidMask[i] = False
                    continue
            else:
                #print("PROTON", i, data.pTrkIdx[i], data.LambdaCd1Idx[i][l1Idx])
                #mcLundi = ak.to_list(mcLund[i][:18])
                #proInd = list(j for j in range(len(mcLundi)) if abs(mcLundi[mothIdx[i][j]])==4122 and abs(mcLundi[j])==2212)
                indexFailList += [(i,"proton", data.pTrkIdx[i], data.LambdaCd1Idx[i][l1Idx])]
                pidMask[i] = False
                continue


        if abs(dd[i])==4:
            if len(data.piTrkIdx[i])>data.LambdaCd2Idx[i][l1Idx]:
                if pimask[i][data.LambdaCd2Idx[i][l1Idx]] == False:
                    pidMask[i] = False
                    continue
            else:
                #print("PION", i, data.piTrkIdx[i], data.LambdaCd2Idx[i][l1Idx])
                #mcLundi = ak.to_list(mcLund[i][:18])
                #piInd = list(j for j in range(len(mcLundi)) if abs(mcLundi[mothIdx[i][j]])==4122 and abs(mcLundi[j])==211)
                indexFailList += [(i,"pion2", data.piTrkIdx[i], data.LambdaCd2Idx[i][l1Idx])]
                pidMask[i] = False
                continue

        if abs(dd[i])!=2:
            if len(data.piTrkIdx[i])>data.LambdaCd3Idx[i][l1Idx]:
                if pimask[i][data.LambdaCd3Idx[i][l1Idx]] == False:
                    pidMask[i] = False
                    continue
            else:
                #print("PION", i, data.piTrkIdx[i], data.LambdaCd3Idx[i][l1Idx])
                #mcLundi = ak.to_list(mcLund[i][:18])
                #piInd = list(j for j in range(len(mcLundi)) if abs(mcLundi[mothIdx[i][j]])==4122 and abs(mcLundi[j])==211)
                indexFailList += [(i,"pion3", data.piTrkIdx[i], data.LambdaCd3Idx[i][l1Idx])]
                pidMask[i] = False
                continue

        if abs(dd[i])==3 or abs(dd[i])==4:
            if len(data.piTrkIdx[i])>data.LambdaCd4Idx[i][l1Idx]:
                if pimask[i][data.LambdaCd4Idx[i][l1Idx]] == False:
                    pidMask[i] = False
                    continue
            else:
                #mcLundi = ak.to_list(mcLund[i][:18])
                #piInd = list(j for j in range(len(mcLundi)) if abs(mcLundi[mothIdx[i][j]])==4122 and abs(mcLundi[j])==211)
                #print("PION", i, data.piTrkIdx[i], data.LambdaCd4Idx[i][l1Idx])
                indexFailList += [(i,"pion4", data.piTrkIdx[i], data.LambdaCd4Idx[i][l1Idx])]
                pidMask[i] = False
                continue

        if abs(dd[i])==1:
            #print(dauL)
            if len(data.KTrkIdx[i])>data.LambdaCd2Idx[i][l1Idx]:
                if kamask[i][data.LambdaCd2Idx[i][l1Idx]] == False:
                    pidMask[i] = False
                    continue
            else:
                #mcLundi = ak.to_list(mcLund[i][:18])
                #kaInd = list(j for j in range(len(mcLundi)) if abs(mcLundi[mothIdx[i][j]])==4122 and abs(mcLundi[j])==321)
                indexFailList += [(i,"Kaon", data.KTrkIdx[i], data.LambdaCd2Idx[i][l1Idx])]
                pidMask[i] = False
                continue


        pidMask[i] = True
    #print("FAIL COUNT", failC)
    return pidMask, indexFailList

In [ ]:
for strengthSP in ["SuperLoose"]:
    strengthSPi = "SuperLoose"
    strengthSK = "SuperLoose"
    myPMask = strengthSP+"KMProtonSelection"
    myPiMask = strengthSPi + "KMPionMicroSelection"
    myKMask = strengthSK + "KMKaonMicroSelection"

    #0.9965

    pidMask, indexFailList = buildPidMask(myPMask, myPiMask, myKMask)

    print(myPMask, myPiMask, myKMask)
    print("TOTAL EVENTS", len(tmBInd))
    print("PASSING EVENTS", sum(1 for val in pidMask.values() if val))
    print("PASS RATE = ", sum(1 for val in pidMask.values() if val)/len(tmBInd))
    print("EVENTS FAILING TRK: ", len(indexFailList))


In [ ]:
postPIDlamC={1:0,2:0,3:0,4:0}
postPIDlamCIndL = []
mask_listpostPID = [[False] * len(sub) for sub in ak.to_list(lamCMassLooseMask)]
for i in tmBInd:
    if pidMask[i]:
        if dd[i]==2 or dd[i]==3:
            j = d2Idx[i][0]
            mask_listpostPID[i][j] = ksFSLooseMask[i][j]&lamCMassLooseMask[i][j]
        else:
            mask_listpostPID[i] = lamCMassLooseMask[i]
    else:
        mask_listpostPID[i]==False
        
ksFScutMassMaskpostPID = ak.Array(mask_listpostPID)
for i in tmBInd:
    postPIDlamC[dd[i]] += ak.sum(ksFScutMassMaskpostPID[i])

print("dm1, tmB, lamC cand:", postPIDlamC[1])
print("dm2, tmB, lamC cand:", postPIDlamC[2])
print("dm3, tmB, lamC cand:", postPIDlamC[3])
print("dm4, tmB, lamC cand:", postPIDlamC[4])
print()

postPIDlam0={1:0,2:0,3:0,4:0}
postPIDlam0IndL = []
for i in tmBInd:
    if pidMask[i]:
        postPIDlam0[dd[i]] += ak.sum(lam0comb[i])

print("dm1, tmB, lam0 cand:", postPIDlam0[1])
print("dm2, tmB, lam0 cand:", postPIDlam0[2])
print("dm3, tmB, lam0 cand:", postPIDlam0[3])
print("dm4, tmB, lam0 cand:", postPIDlam0[4])



In [ ]:
c = 0
numLamC = numLam0 = 0
b1c = 0
pidF=0
b1d = {1:0,2:0,3:0,4:0}
b1dC = {1:0,2:0,3:0,4:0}
for i in tmBInd:
    if pidMask[i]:
        c+=1
        lch= ak.sum(ksFScutMassMaskpostPID[i])
        numLamC+=lch
        l0h=ak.sum(lam0comb[i])
        numLam0+=l0h

        b1d[dd[i]]+=(lch*l0h)

        if lch*l0h==1 or (dd[i]==4 and lch==1 and l0h==2):
            b1dC[dd[i]]+=1
            b1c+=1
    else:
        pidF+=1


In [ ]:
###################################
###################################

######  FINAL OUTPUT SLIDE  #######

###################################
###################################


print("Total events: ", len(data))
print("Truth matched B events:", len(tmBInd))
print("By decay:")
print("Decay Mode 1 tmB events:", oC, " / percent:", oC/len(tmBInd)*100,"%")
print("Decay Mode 2 tmB events:", tC, " / percent:", tC/len(tmBInd)*100,"%")
print("Decay Mode 3 tmB events:", thC, " / percent:", thC/len(tmBInd)*100,"%")
print("Decay Mode 4 tmB events:", fC, " / percent:", fC/len(tmBInd)*100,"%")
print()
print("LamC cand in tmB events by decay:")
print("Decay 1:", dm1lamCcount, " / Decay 2:", dm2lamCcount," / Decay 3:", dm3lamCcount," / Decay 4:", dm4lamCcount)
print("Lam0 cand in tmB events by decay:")
print("Decay 1:", lam0MassWinDprecut[1], " / Decay 2:", lam0MassWinDprecut[2]," / Decay 3:", lam0MassWinDprecut[3]," / Decay 4:", lam0MassWinDprecut[4])
print()
print("Remaining lamC cand after lamC mass cut:")
print("Decay 1:", lamCMassWinD[1], " / Decay 2:", lamCMassWinD[2]," / Decay 3:", lamCMassWinD[3]," / Decay 4:", lamCMassWinD[4])
print("Remaining lamC cand after lamC mass cut and k_s FS cut:")
print("Decay 1:", lamCCandksFSMassCutD[1], " / Decay 2:", lamCCandksFSMassCutD[2]," / Decay 3:", lamCCandksFSMassCutD[3]," / Decay 4:", lamCCandksFSMassCutD[4])
print("Remaining lamC cand after lamC mass cut and k_s FS cut and PID selection:")
print("Decay 1:", postPIDlamC[1], " / Decay 2:", postPIDlamC[2]," / Decay 3:", postPIDlamC[3]," / Decay 4:", postPIDlamC[4])
print("Remaining lam0 cand after lam0 mass cut:")
print("Decay 1:", lam0MassWinD[1], " / Decay 2:", lam0MassWinD[2]," / Decay 3:", lam0MassWinD[3]," / Decay 4:", lam0MassWinD[4])
print("Remaining lam0 cand after lam0 mass cut and lam0 FS cut:")
print("Decay 1:", lam0MassWinDcomb[1], " / Decay 2:", lam0MassWinDcomb[2]," / Decay 3:", lam0MassWinDcomb[3]," / Decay 4:", lam0MassWinDcomb[4])
print("Remaining lam0 cand after lam0 mass cut and lam0 FS cut and PID selection:")
print("Decay 1:", postPIDlam0[1], " / Decay 2:", postPIDlam0[2]," / Decay 3:", postPIDlam0[3]," / Decay 4:", postPIDlam0[4])
print()
print("Events checked:",c, " / Events failing pid:",pidF, " / Total lamC cand:",numLamC, " / Total lam0 cand:",numLam0, " / Total events with 1 B cand:",b1c)
for i in range(1,5):
    print(f"DM {i} / Count of events with 1 B: {b1dC[i]} / Count of this decay mode: {list(dd.values()).count(i)}")
print()





